# Pretrained visual features, PCA and CatBoost

Bundled completed primary-policy development accuracy: **55.0%**, balanced accuracy **54.6%**. Independent test accuracy is unestablished.

This notebook compares 12 fixed candidates on the original 40 recordings with the exact historical outer folds. It includes semantic summaries, all-view CLIP, train-only PCA16, and a full-frame view ablation. A separate X3D-S transfer test retains its pretrained 400-action classifier.

All private perception runs locally in this runtime. Audio, hosted LLM APIs, captions, private notes and unverified custom labels are unused. Related views must share scene groups. Existing custom footage remains unlabeled and is excluded from supervised evaluation.

The embedded snapshot includes 145 implementation regression tests; test counts are not accuracy measurements.

## Restore source and install dependencies
The embedded source archive contains no recordings, frames, weights or feature caches. Public model downloads occur only during setup.

In [ ]:
# Restore exact source bytes; reuse an existing original feature cache in this workspace.
import base64, hashlib, io, json, os, sys, zipfile
from pathlib import Path
BASE = Path('/content') if Path('/content').is_dir() else Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
WORKSPACE = BASE / 'datathon_ml_workspace'
WORKSPACE.mkdir(parents=True, exist_ok=True)
SOURCE_ZIP = base64.b64decode('...')
assert hashlib.sha256(SOURCE_ZIP).hexdigest() == '6865d40d92b714699eda643c6a2ebb475840b75d0ac50c2854cf68d6fd711f9f', 'Embedded source checksum differs'
with zipfile.ZipFile(io.BytesIO(SOURCE_ZIP)) as archive:
    seen = set()
    for member in archive.infolist():
        target = (WORKSPACE / member.filename).resolve()
        if not target.is_relative_to(WORKSPACE.resolve()) or target in seen:
            raise ValueError('Unsafe embedded source path')
        seen.add(target)
    manifest = json.loads(archive.read('SOURCE_MANIFEST.json'))
    for name, digest in manifest['members_sha256'].items():
        assert hashlib.sha256(archive.read(name)).hexdigest() == digest, 'Source member checksum differs'
    archive.extractall(WORKSPACE)
os.chdir(WORKSPACE)
sys.path.insert(0, str(WORKSPACE))
print('Workspace:', WORKSPACE)
print('Source SHA-256:', hashlib.sha256(SOURCE_ZIP).hexdigest())


In [ ]:
import shutil, subprocess, sys

def run_command(arguments):
    subprocess.run(list(map(str, arguments)), cwd=WORKSPACE, check=True)

run_command([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-ml.txt',
    '-r', 'requirements-experiments.txt', 'gdown>=5,<6'])
if not shutil.which('ffprobe'):
    assert Path('/content').is_dir(), 'Install ffmpeg/ffprobe in this runtime first.'
    run_command(['apt-get', '-qq', 'update'])
    run_command(['apt-get', '-qq', 'install', '-y', 'ffmpeg'])
import torch, catboost
assert catboost.__version__ == '1.2.10'
print('Torch:', torch.__version__, 'CatBoost:', catboost.__version__, 'CUDA:', torch.cuda.is_available())
RUN_MOTION = True


In [ ]:
# Public checkpoints/source are cached before any private perception.
for key in ('HF_HUB_OFFLINE', 'TRANSFORMERS_OFFLINE', 'HF_DATASETS_OFFLINE'):
    os.environ.pop(key, None)
from datathon_ml.clip_features import ClipFeatureConfig, _resolve_clip_weights
CLIP_CHECKPOINT = _resolve_clip_weights(ClipFeatureConfig(allow_download=True), torch)
print('CLIP checkpoint cached:', CLIP_CHECKPOINT.is_file())
if RUN_MOTION:
    from datathon_ml.motion import MotionFeatureConfig, _checkpoint, _load_model
    motion_config = MotionFeatureConfig(backbone='x3d_s', representation='logits', device='cpu', allow_download=True)
    X3D_CHECKPOINT = _checkpoint(motion_config, torch)
    cached_motion_model = _load_model(motion_config, X3D_CHECKPOINT, torch.device('cpu'), torch)
    del cached_motion_model
    print('Pinned X3D-S source/checkpoint cached:', X3D_CHECKPOINT.is_file())


## Reuse or build original feature caches
The organizer labels path is explicit. Existing CLIP and semantic caches are reused. Fresh extraction requires a GPU and verifies the original 40 source hashes and labels.

In [ ]:
# An existing complete original image cache is reused. Otherwise bootstrap it.
from datathon_ml.data import build_manifest
from datathon_ml.clip_features import extract_clip_features, ClipFeatureConfig
from datathon_ml.semantic_clip import convert_clip_cache
from tools.benchmark_semantic_vlm import blocked_network, configure_offline_environment
ORIGINAL = WORKSPACE / 'runs/clip_b32'
ORIGINAL_MANIFEST = ORIGINAL / 'manifest.json'
REFERENCE = json.loads((WORKSPACE / 'reference/clip_baseline_cv.json').read_text())
expected = {r['sha256']: r['label'] for r in REFERENCE['predictions']}
assert len(expected) == 40, 'Expected the original 40 unique sources'
original_records = json.loads(ORIGINAL_MANIFEST.read_text()) if ORIGINAL_MANIFEST.is_file() else None
complete = isinstance(original_records, list) and all((ORIGINAL / 'features' / (r['sha256'] + '.npz')).is_file() for r in original_records)
if not complete:
    DATA_ROOT = globals().get('DATA_ROOT', None)
    if DATA_ROOT is not None:
        DATA_ROOT = Path(DATA_ROOT).expanduser().resolve()
    else:
        candidates = [WORKSPACE / 'open', WORKSPACE / 'data/open', BASE / 'open']
        DATA_ROOT = next((p for p in candidates if (p / 'open_labels.xlsx').is_file()), None)
    if DATA_ROOT is None:
        import gdown
        organizer_zip = BASE / 'datathon_original.zip'
        if not organizer_zip.is_file():
            assert gdown.download(id='1H35d4mF1cEUzZYsnViJ0da_PxSmiaiDq', output=str(organizer_zip), quiet=False), 'Organizer download failed'
        destination = WORKSPACE / 'organizer_data'
        destination.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(organizer_zip) as archive:
            for member in archive.infolist():
                target = (destination / member.filename).resolve()
                if not target.is_relative_to(destination.resolve()):
                    raise ValueError('Unsafe organizer archive member')
            archive.extractall(destination)
        matches = list(destination.rglob('open_labels.xlsx'))
        assert len(matches) == 1, 'Select the organizer DATA_ROOT explicitly when several workbooks exist'
        DATA_ROOT = matches[0].parent
    labels_path = DATA_ROOT / 'open_labels.xlsx'
    assert labels_path.is_file(), 'DATA_ROOT must contain the organizer open_labels.xlsx'
    aliases = json.loads((WORKSPACE / 'duplicate_aliases.json').read_text())
    original_records = build_manifest(DATA_ROOT, output_dir=ORIGINAL,
        labels_path=labels_path, verified_aliases=aliases)
    assert {r['sha256']: r['label'] for r in original_records} == expected, 'Organizer sources/labels differ from the frozen reference'
    configure_offline_environment()
    with blocked_network() as bootstrap_network:
        extract_clip_features(original_records, ORIGINAL / 'features',
            ClipFeatureConfig(device='cuda', batch_size=32, allow_download=False))
    assert bootstrap_network['attempts'] == 0
    print('Original image extraction network attempts:', bootstrap_network['attempts'])
assert {r['sha256']: r['label'] for r in original_records} == expected, 'Cached originals do not match the frozen reference'
SEMANTIC_ORIGINAL = WORKSPACE / 'runs/semantic_temporal_v1/semantic_clip/features'
# Reuses exact existing prototypes; a fresh workspace constructs them on CPU,
# matching the historical original semantic prototype protocol.
configure_offline_environment()
with blocked_network() as semantic_network:
    convert_clip_cache(ORIGINAL / 'features', SEMANTIC_ORIGINAL, device='cpu', offline=True)
assert semantic_network['attempts'] == 0
print('Original unique sources:', len(original_records))
print('Original semantic extraction network attempts:', semantic_network['attempts'])


In [ ]:
# Every experimental command runs in a fresh process under a socket guard.
import time
RUN = WORKSPACE / 'runs/boosted_v1'
RUN.mkdir(parents=True, exist_ok=True)
GUARD_WRAPPER = r"""
import json, runpy, sys, time
from pathlib import Path
module, output, arguments = sys.argv[1:]
from tools.benchmark_semantic_vlm import configure_offline_environment, blocked_network
configure_offline_environment()
started = time.perf_counter()
result = {'status': 'started', 'completed': False}
sys.argv = [module + '.py'] + json.loads(arguments)
with blocked_network() as network:
    try:
        runpy.run_module(module, run_name='__main__')
        result.update(status='completed', completed=True)
    except SystemExit as error:
        if error.code in (None, 0):
            result.update(status='completed', completed=True)
        else:
            result.update(status='failed', error_type='SystemExit')
        raise
    except BaseException as error:
        result.update(status='failed', error_type=type(error).__name__)
        raise
    finally:
        result.update(network_attempts=network['attempts'], elapsed_seconds=time.perf_counter()-started)
        Path(output).write_text(json.dumps(result, indent=2))
        print('GUARDED_RUNTIME', json.dumps(result), flush=True)
"""

def run_guarded(module, arguments, runtime_name, budget):
    runtime = RUN / runtime_name
    log = runtime.with_suffix('.log')
    env = os.environ.copy()
    env.update(PYTHONUNBUFFERED='1', PYTHONHASHSEED='42', OMP_NUM_THREADS='2',
               OPENBLAS_NUM_THREADS='2', MKL_NUM_THREADS='2')
    started = time.perf_counter()
    with log.open('w') as output:
        process = subprocess.Popen([sys.executable, '-u', '-c', GUARD_WRAPPER, module,
            str(runtime), json.dumps(list(map(str, arguments)))], cwd=WORKSPACE,
            stdout=output, stderr=subprocess.STDOUT, env=env)
        try:
            code = process.wait(timeout=budget)
        except (subprocess.TimeoutExpired, KeyboardInterrupt) as error:
            process.kill()
            process.wait()
            report = {'status': 'timed_out' if isinstance(error, subprocess.TimeoutExpired) else 'cancelled',
                      'completed': False, 'network_attempts': None, 'budget_seconds': budget,
                      'elapsed_seconds': time.perf_counter()-started}
            runtime.write_text(json.dumps(report, indent=2))
            print(report)
            raise
    report = json.loads(runtime.read_text()) if runtime.is_file() else {'status': 'failed_without_runtime_report'}
    report.update(returncode=code, budget_seconds=budget)
    runtime.write_text(json.dumps(report, indent=2))
    print('Runtime:', report, 'Log:', log)
    if code:
        raise RuntimeError('Guarded command failed; inspect its local log.')
    assert report.get('network_attempts') == 0, 'Unexpected network attempt'
    return report


## Validate implementation and run the fixed comparison
CatBoost uses 160 iterations, learning rate 0.03, depth 2/3 and L2 10/30. Inner grouped OOF selects representation, tree settings and threshold; every scaler/PCA fit uses training sources only. No outer early stopping or calibration occurs. The full comparison has an 1,800-second hard budget.

In [ ]:
run_guarded('unittest', ['discover', '-s', 'tests', '-v'], 'runtime_tests.json', 900)


In [ ]:
run_guarded('boosted_experiment', ['compare',
    '--original-manifest', ORIGINAL_MANIFEST, '--clip-features', ORIGINAL / 'features',
    '--semantic-features', SEMANTIC_ORIGINAL, '--folds-from', 'reference/clip_baseline_cv.json',
    '--output-dir', RUN, '--threads', '2', '--fit-final'], 'runtime_boosted.json', 1800)


## Separate pretrained action transfer test
X3D-S processes ordered clips through frozen pretrained weights, retaining all 400 action logits. A recording head pools mean/std/max and selects among the same three tree settings. This is a separate preregistered development comparison with its own 1,800-second budget.

In [ ]:
if RUN_MOTION:
    assert torch.cuda.is_available(), 'Select a GPU runtime for fresh X3D extraction.'
    run_guarded('motion_action_experiment', ['run', '--original-manifest', ORIGINAL_MANIFEST,
        '--features', RUN / 'x3d_logits_features', '--output-dir', RUN / 'motion_action',
        '--folds-from', 'reference/clip_baseline_cv.json', '--device', 'cuda', '--fit-final'],
        'runtime_motion_action.json', 1800)
else:
    (RUN / 'runtime_motion_action.json').write_text(json.dumps({'status': 'not_run', 'completed': False,
                                                            'network_attempts': None}))


## Offline prediction, diagnostics and measured comparison
Recording heads provide binary recording decisions. Event timestamp recall is unavailable, not zero. The chart uses a full 0–100% scale. All completed OOF outputs are checked against the frozen SHA, label and fold reference, with nested training ownership and model source fingerprints verified.

In [ ]:
# Features are extracted from video alone; no labels or points enter prediction.
key = sorted(r['sha256'] for r in original_records)[0]
run_guarded('boosted_experiment', ['predict', '--model', RUN / 'model/model_boosted.joblib',
    '--clip-feature', ORIGINAL / 'features' / (key + '.npz'),
    '--semantic-feature', SEMANTIC_ORIGINAL / (key + '.npz')], 'runtime_inference.json', 120)
if RUN_MOTION and (RUN / 'motion_action/model/model_motion_action.joblib').is_file():
    run_guarded('motion_action_experiment', ['predict', '--model', RUN / 'motion_action/model/model_motion_action.joblib',
        '--feature-path', RUN / 'x3d_logits_features' / (key + '.npz')], 'runtime_motion_inference.json', 120)


In [ ]:
# PCA diagnostics use training sources only; variance is not model accuracy.
run_guarded('tools.audit_preprocessing', ['--original-manifest', ORIGINAL_MANIFEST,
    '--features', ORIGINAL / 'features', '--folds-from', 'reference/clip_baseline_cv.json',
    '--output', RUN / 'preprocessing_audit.json'], 'runtime_preprocessing.json', 600)
run_command([sys.executable, 'tools/summarize_boosted.py', '--results-root', RUN,
    '--baseline', 'reference/clip_baseline_cv.json',
    '--previous-summary', 'reference/previous_custom_summary.json', '--output-dir', 'artifacts'])
summary = json.loads((WORKSPACE / 'artifacts/boosted_experiment_summary.json').read_text())
print('MODEL / ACCURACY / BALANCED ACCURACY / RECALL / SPECIFICITY')
for row in summary['model_comparisons']:
    m = row['metrics']
    print(row['model'], *(f"{m[k]:.1%}" for k in ('accuracy', 'balanced_accuracy', 'recall', 'specificity')))
print('87% target reached on development:', summary['target_achieved_on_development'])
print('Independent 87% test accuracy established:', summary['independent_87_percent_accuracy_established'])
from IPython.display import display, Image
display(Image(filename=str(WORKSPACE / 'artifacts/boosted_model_comparison.png')))
measured_reference = WORKSPACE / 'reference/measured_boosted_summary.json'
if measured_reference.is_file():
    print('Bundled immutable measured reference rows:', len(json.loads(measured_reference.read_text())['model_comparisons']))


## Export reproducible artifacts
The export includes trained models, numeric reports and exact source bytes. Private footage, audio, extracted images, feature caches and private notes are excluded. Public data candidates and suitability/license findings are documented in `reference/public_video_data_research.json`; no external training footage has been silently added.

In [ ]:
# Export exact numeric reports, trained artifacts and checksummed source only.
RESULTS_ZIP = BASE / 'datathon_boosted_results.zip'
candidates = list(RUN.glob('*.json')) + list((RUN / 'motion_action').glob('*.json'))
candidates += list((RUN / 'model').glob('*.joblib')) + list((RUN / 'model').glob('*.json'))
candidates += list((RUN / 'motion_action/model').glob('*.joblib')) + list((RUN / 'motion_action/model').glob('*.json'))
candidates += [WORKSPACE / 'artifacts' / name for name in ('boosted_experiment_summary.json',
               'boosted_model_comparison.csv', 'boosted_model_comparison.png')]
candidates += [WORKSPACE / 'datathon_boosted.ipynb']
members = {}
with zipfile.ZipFile(io.BytesIO(SOURCE_ZIP)) as source:
    for name in source.namelist():
        members['source/' + name] = source.read(name)
for path in candidates:
    if path.is_file():
        name = path.relative_to(WORKSPACE).as_posix()
        assert not (set(Path(name).parts) & {'features', 'x3d_logits_features', 'private_training_notes', 'data', 'open'})
        assert path.suffix.lower() in {'.json', '.csv', '.png', '.joblib', '.ipynb'}
        if path.suffix.lower() == '.ipynb':
            clean_notebook = json.loads(path.read_text())
            assert all(not cell.get('outputs') and cell.get('execution_count') is None
                       for cell in clean_notebook['cells'] if cell['cell_type'] == 'code')
        members[name] = path.read_bytes()
members['EXPORT_MANIFEST.json'] = json.dumps({'format_version': 1, 'contains_private_media_or_feature_caches': False,
    'members_sha256': {name: hashlib.sha256(value).hexdigest() for name, value in sorted(members.items())}}, indent=2).encode()
with zipfile.ZipFile(RESULTS_ZIP, 'w', zipfile.ZIP_DEFLATED) as archive:
    for name, value in sorted(members.items()):
        archive.writestr(name, value)
print('Export:', RESULTS_ZIP, 'Bytes:', RESULTS_ZIP.stat().st_size)
print('SHA256:', hashlib.sha256(RESULTS_ZIP.read_bytes()).hexdigest())
# Download privately when ready:
# from google.colab import files
# files.download(str(RESULTS_ZIP))
